<a href="https://colab.research.google.com/github/wbc-mjlab/wbc-mjlab/blob/main/notebooks/custom_task_train.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Custom G1 WBC task + short train

This notebook shows the **tasks, not forks** pattern:

1. Duplicate the `Wbc-G1` env builder (rewards, RSI, terminations — see [`wbc.py`](../src/wbc_mjlab/robots/g1/configs/wbc.py))
2. Change a few weights / RSI knobs in `g1_custom_wbc_env_cfg()`
3. Register `Wbc-G1-Custom` inline (patch `G1_TASK_BY_ID`, then `register_wbc_task`)
4. Train on **samples** with `--agent.resume True` so you can stop and continue

For production, move the builder into `src/wbc_mjlab/robots/g1/configs/` and add it to `G1_WBC_TASKS`.

> **Runtime:** GPU (T4+). Re-run the train cell to resume from the latest checkpoint in `logs/rsl_rl/wbc_g1_custom/`.

## 1. Setup

In [ ]:
import os

os.environ["MUJOCO_GL"] = "egl"

REPO_ROOT = "/content/wbc-mjlab"

if not os.path.isdir(REPO_ROOT):
  !git clone -q https://github.com/wbc-mjlab/wbc-mjlab.git {REPO_ROOT}

%cd {REPO_ROOT}

!curl -LsSf https://astral.sh/uv/install.sh | sh
os.environ["PATH"] = os.environ["HOME"] + "/.local/bin:" + os.environ["PATH"]

!uv sync --extra cu128
!nvidia-smi
print("✓ wbc-mjlab ready")

## 2. Convert samples

In [ ]:
!uv run wbc-mjlab-data-to-npz --robot g1 --dataset samples --batch-size 16

## 3. Define a custom env config

Explicit copy of [`wbc.py`](../src/wbc_mjlab/robots/g1/configs/wbc.py) so reward weights, RSI, and terminations are visible. We only change **action-rate** and **RSI bin width** at the bottom.

In [ ]:
from dataclasses import replace

from mjlab.envs import ManagerBasedRlEnvCfg
from mjlab.managers.termination_manager import TerminationTermCfg

import wbc_mjlab.env.mdp as mdp
from wbc_mjlab.env.mdp.commands import MotionCommandCfg
from wbc_mjlab.robots.g1.configs.base import (
  G1_EE_TERMINATION_BODY_NAMES,
  G1_MOTION_BODY_NAMES,
  g1_base_cfg,
)

# Zest Table S4: exp(-κ‖e‖²/σ²) with κ = 1/4.
_TRACKING_KAPPA = 0.25


def g1_custom_wbc_env_cfg() -> ManagerBasedRlEnvCfg:
  """Wbc-G1 variant: same Zest S4 stack as wbc.py, with two Colab tweaks at the end."""
  cfg = g1_base_cfg()
  rw = cfg.rewards

  # --- Zest Table S4 tracking (all weight 1.0) ---
  rw["motion_global_root_pos"].weight = 1.0
  rw["motion_global_root_pos"].params["std"] = 0.4
  rw["motion_global_root_ori"].weight = 1.0
  rw["motion_global_root_ori"].params["std"] = 0.5
  rw["motion_root_lin_vel_b"].weight = 1.0
  rw["motion_root_lin_vel_b"].params["std"] = 0.6
  rw["motion_root_ang_vel_b"].weight = 1.0
  rw["motion_root_ang_vel_b"].params["std"] = 1.5
  rw["motion_body_pos"].weight = 1.0
  rw["motion_body_pos"].params.pop("std", None)
  rw["motion_body_pos"].params["sigma_per_keybody"] = 0.2
  rw["motion_body_pos"].params["body_error_aggregate"] = "sum"
  rw["motion_body_pos"].params["body_names"] = G1_MOTION_BODY_NAMES
  rw["motion_body_ori"].weight = 1.0
  rw["motion_body_ori"].params.pop("std", None)
  rw["motion_body_ori"].params["sigma_per_keybody"] = 0.4
  rw["motion_body_ori"].params["body_error_aggregate"] = "sum"
  rw["motion_body_ori"].params["body_names"] = G1_MOTION_BODY_NAMES
  rw["motion_joint_pos"].weight = 1.0
  rw["motion_joint_pos"].params.pop("std", None)
  rw["motion_joint_pos"].params.pop("per_joint", None)
  rw["motion_joint_pos"].params["sigma_per_joint"] = 0.3
  for name in (
    "motion_global_root_pos",
    "motion_global_root_ori",
    "motion_root_lin_vel_b",
    "motion_root_ang_vel_b",
    "motion_body_pos",
    "motion_body_ori",
    "motion_joint_pos",
  ):
    rw[name].params["kappa"] = _TRACKING_KAPPA

  # --- WBC extras ---
  rw["motion_body_lin_vel"].weight = 0.5
  rw["motion_body_lin_vel"].params["std"] = 1.0
  rw["motion_body_lin_vel"].params["body_names"] = G1_MOTION_BODY_NAMES
  rw["motion_body_ang_vel"].weight = 0.5
  rw["motion_body_ang_vel"].params["std"] = 3.14
  rw["motion_body_ang_vel"].params["body_names"] = G1_MOTION_BODY_NAMES
  rw["motion_joint_vel"].weight = 0.0
  rw["neg_regen_power"].weight = 0.0
  rw["angular_momentum"].weight = 0.0

  rw["survival"].weight = 1.0
  rw["action_rate_l1"].weight = -0.2  # default Wbc-G1 uses -0.1
  rw["joint_acc"].weight = -5.0e-6
  rw["joint_limit"].weight = -1.0
  rw["actuator_torque_soft_limit"].weight = -0.1
  rw["actuator_torque_soft_limit"].params["soft_ratio"] = 0.9

  rw["foot_slip"].weight = -0.0
  rw["anti_shake"].weight = 0.0

  cfg.observations["actor"].history_length = 1
  cfg.observations["actor"].terms.pop("ref_joint_vel", None)

  motion_cmd = cfg.commands["motion"]
  assert isinstance(motion_cmd, MotionCommandCfg)
  motion_cmd.assistive_wrench_enabled = True
  motion_cmd.rsi = replace(
    motion_cmd.rsi,
    similarity_from_rewards=True,
    bin_width_s=2.0,  # default Wbc-G1 uses 4.0
    similarity_norm_by_remaining_clip=True,
    min_bin_span_ratio=0.5,
    persist_failure_levels=True,
  )

  cfg.terminations["anchor_pos"].params["threshold"] = 0.35
  cfg.terminations["ee_body_pos"] = TerminationTermCfg(
    func=mdp.bad_motion_body_pos_z_only,
    params={
      "command_name": "motion",
      "threshold": 0.25,
      "body_names": G1_EE_TERMINATION_BODY_NAMES,
    },
  )
  cfg.terminations["keybody_ground_contact_force"] = TerminationTermCfg(
    func=mdp.excessive_keybody_ground_contact_force,
    params={
      "sensor_name": "keybodies_ground_contact",
      "body_names": G1_MOTION_BODY_NAMES,
      "body_slot_order": G1_MOTION_BODY_NAMES,
      "force_threshold": 2000.0,
    },
  )

  return cfg

### Register at runtime (notebook only)

In the repo you would add the task to `G1_WBC_TASKS` in `configs/__init__.py`. Here we patch the G1 task table and call `register_wbc_task` directly.

In [ ]:
from wbc_mjlab import tasks as wbc_tasks
from wbc_mjlab.robots.g1.configs import G1_TASK_BY_ID
from wbc_mjlab.tasks import list_wbc_task_ids, register_wbc_task
from wbc_mjlab.tasks.config import WbcTaskConfig

CUSTOM_TASK = WbcTaskConfig(
  task_id="Wbc-G1-Custom",
  robot_id="g1",
  description="Colab example: tighter RSI bins + stronger action-rate penalty.",
  experiment_name="wbc_g1_custom",
  build_env_cfg=g1_custom_wbc_env_cfg,
)

G1_TASK_BY_ID[CUSTOM_TASK.task_id] = CUSTOM_TASK
wbc_tasks._ensure_tasks()
wbc_tasks._TASK_BY_ID[CUSTOM_TASK.task_id] = CUSTOM_TASK
register_wbc_task(CUSTOM_TASK)

print("Registered:", CUSTOM_TASK.task_id)
print("All task ids:", list_wbc_task_ids())

## 4. Train (short Colab run)

Logs: `logs/rsl_rl/wbc_g1_custom/`. `--agent.resume True` loads the latest checkpoint in that folder — interrupt the cell and re-run to continue training.

In [ ]:
import sys

from wbc_mjlab.scripts.train import main as train_main

# In-process so the runtime task registration above is visible to prepare_wbc_run.
sys.argv = [
  "wbc-mjlab-train",
  "--task", "Wbc-G1-Custom",
  "--dataset", "samples",
  "--env.scene.num-envs", "4096",
  "--agent.max-iterations", "1000",
  "--agent.resume", "True",
]
train_main()

## 5. Play latest checkpoint (optional)

Re-run section 3 if you restarted the kernel. Auto-detects the newest run under `logs/rsl_rl/wbc_g1_custom/`.

In [ ]:
import sys
import threading

from wbc_mjlab.scripts.play import main as play_main

VISER_PORT = 8081

# In-process (same kernel as registration). Blocks until Viser is up.
sys.argv = [
  "wbc-mjlab-play",
  "--task", "Wbc-G1-Custom",
  "--dataset", "samples",
  "--viewer", "viser",
  "--num-envs", "4",
]

def _run():
  try:
    play_main()
  except Exception as exc:
    print(exc, file=sys.stderr)

threading.Thread(target=_run, daemon=True).start()
print(f"✅ Viser starting on port {VISER_PORT} — run the iframe cell below when ready.")

In [ ]:
from google.colab import output

output.serve_kernel_port_as_iframe(VISER_PORT)